In [0]:
from pyspark.sql import functions as F

df = spark.read.table("dbr_dev_ua5816bd.team_tristar_bronze.routes")

df = df.withColumn(
    'route_id',
    F.when(
        F.col('route_id').try_cast('int').isNull(),
        None
    ).otherwise(
        F.col('route_id').try_cast('int')
    )
)

df = df.withColumn(
    'agency_id',
    F.when(
        F.col('agency_id').try_cast('int').isNull(),
        None
    ).otherwise(
        F.col('agency_id').try_cast('int')
    )
)

df = df.withColumn(
    'route_short_name',
    F.when(
        F.trim(F.col('route_short_name')) == '',
        None
    ).otherwise(
        F.trim(F.col('route_short_name'))
    )
)

df = df.withColumn(
    'route_long_name',
    F.when(
        F.trim(F.col('route_long_name')) == '',
        None
    ).otherwise(
        F.trim(F.col('route_long_name'))
    )
)

df = df.withColumn(
    'route_desc',
    F.when(
        F.trim(F.col('route_desc')) == '',
        None
    ).otherwise(
        F.trim(F.col('route_desc'))
    )
)

df = df.withColumn(
    'route_type',
    F.when(
        ~F.col('route_type').try_cast('int').isin(700, 900, 1200),
        None
    ).otherwise(
        F.col('route_type').try_cast('int')
    )
)

df = df.withColumn(
    'route_color',
    F.when(
        F.trim(F.col('route_color')) == '',
        None
    ).otherwise(
        F.upper(F.trim(F.col('route_color')))
    )
)

df = df.withColumn(
    'route_text_color',
    F.when(
        F.trim(F.col('route_text_color')) == '',
        None
    ).otherwise(
        F.upper(F.trim(F.col('route_text_color')))
    )
)

df = df.withColumn(
    'source',
    F.when(
        F.trim(F.col('source')) == '',
        None
    ).otherwise(
        F.trim(F.col('source'))
    )
)

df = df.withColumn(
    'source_update_date',
    F.when(
        F.trim(F.col('source_update_date')) == '',
        None
    ).otherwise(
        F.trim(F.col('source_update_date'))
    )
)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "dbr_dev_ua5816bd.team_tristar_silver.routes"
)

silver_table.alias("silver").merge(
    df.alias("bronze"),
    "silver.route_id = bronze.route_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).whenNotMatchedBySourceDelete(
).execute()